#  Hotel Booking Data Analysis
### H1 (Resort Hotel) + H2 (City Hotel)
---

## 1. مقدمة المشروع (Project Overview)

**الهدف:** تحليل بيانات حجوزات الفنادق لفهم أنماط سلوك العملاء وتحديد العوامل المؤثرة على الإلغاء، بهدف تقديم توصيات استراتيجية لتحسين الإيرادات والعمليات التشغيلية.

**المشكلة:** عدم وجود عمود موحد يجمع نوع الفندق، ووجود بيانات الحجوزات في ملفين منفصلين (H1, H2).

**الحل:** إضافة عمود `hotel` لكل ملف لتمييز نوع الفندق (Resort vs City)، ثم دمج الملفين في DataFrame واحد لسهولة التحليل الشامل.

## 2.  Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
print('✅ Libraries imported successfully')

In [ ]:
h1 = pd.read_csv('H1.csv')
h2 = pd.read_csv('H2.csv')

h1['hotel'] = 'Resort Hotel'
h2['hotel'] = 'City Hotel'

df = pd.concat([h1, h2], ignore_index=True)
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include="object")

**تحليل الموسمية (Seasonality Insight)**
من جدول الـ object نجد أن العمود `ArrivalDateMonth` قيمته الأكثر تكراراً (top) هي **August**.

>  **Insight:** شهر أغسطس هو شهر الذروة في الحجوزات. هذا يعني أن الفندق يحتاج لرفع الجاهزية التشغيلية (طاقم عمل، صيانة، إمدادات) في هذا الشهر تحديداً لضمان تجربة ممتازة للضيوف.

**تحليل السوق (Market Insight)**
في عمود `Country` نجد أن القيمة الأكثر تكراراً هي **PRT (البرتغال)**.

>  **Insight:** معظم الضيوف يأتون من البرتغال، مما يجعلها السوق الأساسي للفندق. يجب أن تركز الحملات التسويقية والعروض الخاصة على هذا الجمهور لتعظيم الأرباح.

**تحليل سياسات الدفع (Payment Policy Insight)**
في عمود `DepositType` نجد أن **No Deposit** هي الأكثر تكراراً بـ 104,641 حجز.

>  **Insight:** غالبية الحجوزات تتم بدون إيداع مسبق، وهذا قد يفسر ارتفاع نسبة الإلغاء المحتملة.

**تحليل قنوات التوزيع (Distribution Channels Insight)**
قناة **TA/TO** (وكالات السفر) هي القناة الأكثر استخداماً بـ 97,870 حجز.

>  **Insight:** الفندق يعتمد بشكل أساسي على وكالات السفر في جلب الضيوف. العلاقات مع هذه الوكالات حيوية جداً لاستمرارية العمل.

## 3.  Missing Values Analysis

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})

print("Columns with missing values:")
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# df.isna() - show boolean table (sample)
df.isna().head(10)

## 4.  Business Questions & Analysis

### Q1 — What is the average lead time?

In [ ]:
average_lead_time = df['LeadTime'].mean()
print(f"Average Lead Time is: {average_lead_time:.2f} days")

>  **Insight:** بما إن العميل بيحجز قبل الإقامة بـ 104 يوم تقريباً (أكثر من 3 شهور)، ده معناه إننا بنتعامل مع عملاء بيخططوا لسفرهم بدري جداً. فالمفروض نعمل عروض **'Early Bird'** (خصم للحجز المبكر) ونكون حذرين، لأن اللي بيحجز بدري احتمالية إنه يلغي حجزه مع الوقت بتبقى أكبر.

### Q2 — Which hotel type receives more bookings?

In [ ]:
hotel_counts = df['hotel'].value_counts()
hotel_percentages = df['hotel'].value_counts(normalize=True) * 100

print("Number of bookings per hotel type:")
print(hotel_counts)
print("\nPercentage of bookings per hotel type:")
print(hotel_percentages)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = ['#2196F3', '#FF7043']
hotel_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Bookings per Hotel Type', fontsize=13)
axes[0].set_xlabel('Hotel Type')
axes[0].set_ylabel('Number of Bookings')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Pie chart
axes[1].pie(hotel_counts, labels=hotel_counts.index,
            autopct='%1.1f%%', colors=colors,
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Hotel Type Distribution', fontsize=13)

plt.tight_layout()
plt.savefig('plot_hotel_type.png', dpi=150, bbox_inches='tight')
plt.show()

### Q3 — Which months have the highest reservations?

In [ ]:
months_order = ['January', 'February', 'March', 'April', 'May', 'June',
                'July', 'August', 'September', 'October', 'November', 'December']

monthly_bookings = df['ArrivalDateMonth'].value_counts().reindex(months_order)

print("Monthly Reservations:")
print(monthly_bookings)

best_month = monthly_bookings.idxmax()
worst_month = monthly_bookings.idxmin()
print(f"\nThe month with the highest reservations is: {best_month}")
print(f"The month with the lowest reservations is:  {worst_month}")

In [ ]:
palette = sns.color_palette('RdYlGn', 12)
colors_month = [palette[i] for i in range(12)]

plt.figure(figsize=(13, 5))
bars = plt.bar(monthly_bookings.index, monthly_bookings.values,
               color=colors_month, edgecolor='black')
plt.title('Monthly Reservations', fontsize=14)
plt.ylabel('Number of Bookings')
plt.xlabel('Month')
plt.xticks(rotation=45)
for bar, val in zip(bars, monthly_bookings.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{val:,}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('plot_monthly.png', dpi=150, bbox_inches='tight')
plt.show()

>  **Insight:** أظهرت البيانات تباينًا واضحًا في معدلات الحجوزات على مدار السنة؛ حيث يُعد شهر **أغسطس** هو شهر الذروة بأعلى عدد حجوزات (13,877)، بينما يُعد شهر **يناير** هو الأقل نشاطًا (5,929).
>
> - **خلال فترة الذروة (أغسطس):** التركيز على تعظيم العائد عبر استراتيجيات تسعير مرنة، وضمان الجاهزية التشغيلية القصوى.
> - **خلال فترة الركود (يناير):** إطلاق عروض ترويجية جاذبة واستغلال الفترة لأعمال الصيانة.

### Q4 — What factors affect cancellations?

In [ ]:
cancellation_correlation = df.corr(numeric_only=True)['IsCanceled'].sort_values(ascending=False)

print("--- Correlation of Cancellations with other factors ---")
print(cancellation_correlation)

In [ ]:
corr_data = cancellation_correlation.drop('IsCanceled')

colors_corr = ['#F44336' if v > 0 else '#4CAF50' for v in corr_data.values]

plt.figure(figsize=(10, 6))
bars = plt.barh(corr_data.index, corr_data.values, color=colors_corr, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Correlation with IsCanceled', fontsize=14)
plt.xlabel('Correlation Coefficient')
for bar, val in zip(bars, corr_data.values):
    plt.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
plt.tight_layout()
plt.savefig('plot_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

>  **Insight:**
> - **عوامل تزيد من احتمالية الإلغاء (ارتباط موجب):** العامل الأبرز هو `LeadTime` (0.29)، يليه `PreviousCancellations` (0.11).
> - **عوامل تقلل من احتمالية الإلغاء (ارتباط سالب):** `TotalOfSpecialRequests` (-0.23) و`RequiredCarParkingSpaces` (-0.19) و`BookingChanges` (-0.14).
>
> **توصية:** تطبيق سياسات إيداع أكثر صرامة للحجوزات ذات الـ LeadTime المرتفع، وتشجيع الضيوف على تقديم طلبات خاصة أثناء الحجز.

### Q5 — Which countries generate the largest number of bookings?

In [ ]:
country_bookings = df['Country'].value_counts()
top_countries = country_bookings.head(10)

print("Top 10 Countries with the most bookings:")
print(top_countries)

In [ ]:
plt.figure(figsize=(11, 5))
bars = plt.bar(top_countries.index, top_countries.values,
               color=sns.color_palette('Blues_r', 10), edgecolor='black')
plt.title('Top 10 Countries by Number of Bookings', fontsize=14)
plt.ylabel('Number of Bookings')
plt.xlabel('Country Code')
for bar, val in zip(bars, top_countries.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{val:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('plot_countries.png', dpi=150, bbox_inches='tight')
plt.show()

>  **Insight:** البرتغال (PRT) تتصدر القائمة بـ 48,590 حجز، تليها المملكة المتحدة (GBR) بـ 12,129، ثم فرنسا (FRA) بـ 10,415.
>
> - توجيه الجزء الأكبر من الميزانية التسويقية للحفاظ على السوق البرتغالي.
> - تحسين العروض الموجهة لبريطانيا وفرنسا وألمانيا لزيادة الإيرادات من هذه الأسواق.

### Q6 — What is the percentage of repeated guests?

In [ ]:
repeat_guests = df['IsRepeatedGuest'].value_counts(normalize=True) * 100

print("--- Repeat Guest Percentage ---")
print(f"Non-Repeated Guests: {repeat_guests[0]:.2f}%")
print(f"Repeated Guests:     {repeat_guests[1]:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
labels = ['Non-Repeated Guests', 'Repeated Guests']
sizes = [repeat_guests[0], repeat_guests[1]]
colors_pie = ['#42A5F5', '#FF7043']
wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.1f%%',
                                   colors=colors_pie, startangle=90,
                                   wedgeprops={'edgecolor':'white','linewidth':2})
for at in autotexts:
    at.set_fontsize(12)
ax.set_title('Repeated vs Non-Repeated Guests', fontsize=14)
plt.tight_layout()
plt.savefig('plot_repeat_guests.png', dpi=150, bbox_inches='tight')
plt.show()

>  **Insight:** قاعدة عملائنا تعتمد بشكل أساسي على العملاء الجدد بنسبة **96.8%**، في حين لا تتجاوز نسبة العملاء المتكررين **3.2%**.
>
> **توصية:** تبني برامج ولاء (Loyalty Programs) تستهدف الضيوف بعد إقامتهم الأولى وتقديم عروض خاصة لتشجيعهم على العودة، حيث أن تكلفة الحفاظ على عميل حالي أقل بكثير من تكلفة جذب عميل جديد.

## 5.  النتائج الرئيسية والتوصيات النهائية

### Key Findings
| الموضوع | النتيجة |
|---------|---------|
| توزيع الطلب | فنادق المدينة (City Hotels) تستحوذ على **66.4%** من الحجوزات |
| الموسمية | **أغسطس** ذروة الحجوزات — **يناير** الأقل نشاطاً |
| عوامل الإلغاء | الـ **Lead Time** هو أقوى مؤشر للإلغاء (r=0.29) |
| الأسواق | **البرتغال** السوق الأول، تليها بريطانيا وفرنسا |
| ولاء العملاء | **3.2%** فقط من العملاء متكررون |

### Business Recommendations
1. **استراتيجية التسويق:** توجيه الميزانية لدعم الأسواق الدولية (بريطانيا وفرنسا) بجانب السوق البرتغالي.
2. **إدارة الإلغاء:** تشديد سياسات الإيداع للحجوزات ذات الـ Lead Time المرتفع.
3. **تحسين الولاء:** إطلاق برامج ولاء لتحفيز الضيوف على تكرار الزيارة.